# The Beatles, Part B: Tidy Data

**Beatles notebooks:** [A. Clean Data](A_Beatles_Clean_Data.ipynb) · **B. Tidy Data** · [C. Find, Filter, Group](C_Beatles_Find_Filter_Group.ipynb) · [D. Charts and Graphs](D_Beatles_Charts_Graphs.ipynb)

**The big picture.** A cell that reads `Psychedelic Rock, Art Rock, Pop/Rock` is easy for a person to read but hard for a computer to count. How many Beatles songs are Art Rock? Which songwriter favored Folk Rock? To answer questions like these, each label has to stand on its own. **Tidy data** is a short set of principles for arranging a table so that such questions become a single line of code:

1. each **variable** forms a column
2. each **observation** forms a row
3. each type of observational unit forms a table

(From Hadley Wickham's influential article, ["Tidy Data"](https://www.jstatsoft.org/article/view/v059i10).)

**In this notebook you will learn to:**
* recognize untidy data in our Beatles table
* split one column holding two variables into two columns (UK vs. US album debuts)
* `explode` a list of labels into one row per label (genres)
* `melt` "wide" data into "long" data, the format most charting tools prefer
* weigh the trade-offs: why tidying *everything* at once is a bad idea, and how to count songs correctly in tidy data

**Input → Output:** `beatles_clean.pkl` (from Notebook A) → **`beatles_tidy_genre.pkl`** (one row per song × genre). Run Notebook A first!

**Keep in mind:** "tidy" always depends on the question. A table that is tidy for studying *genres* (one row per genre) is awkward for studying *songs* (one row per song). That is why we keep both versions.

**Contents of this Notebook**

- **0.** Import Libraries
- **1.** Tidy Data Principles
- **2.** Multiple Variables in One Column: UK and US Album Debuts
- **3.** Multiple Observations in One Row: Genre (`explode`)
- **4.** Caution:  Why Not Explode Everything?
- **5.** Wide to Long: Audio Features (`melt`)
- **6.** Save the Tidy Data as a Pickle

## 0. Import Libraries

In [16]:
import pandas as pd
pd.set_option('display.max_columns', None)

beatles_clean = pd.read_pickle('beatles_clean.pkl')
beatles_clean.shape

(278, 49)

> If `beatles_clean.pkl` is missing, run **Notebook A_Beatles_Clean_data.ipynb** first: it creates the file.

## 1. Tidy Data Principles

1. **Each variable forms a column.** In our data, `album_debut` sometimes holds *two* variables in one cell: the UK album and the US album (`UK: With the Beatles US: Meet The Beatles!`).
2. **Each observation forms a row.** The `genre` column holds *several* observations in one cell (`Psychedelic Rock, Art Rock, Pop/Rock` means "this song is Psychedelic Rock," "this song is Art Rock," ...).
3. **Each type of observational unit forms a table.** Our table is about *songs*. If we wanted to study *albums* or *musicians*, a separate table would be tidier.

Let's see the problem columns:

In [17]:
beatles_clean[['title', 'album_debut', 'genre']].iloc[[1, 2, 4]]

,title,album_debut,genre
1,A Day in the Life,Sgt. Pepper's Lonely Hearts Club Band,"[Psychedelic Rock, Art Rock, Pop/Rock]"
2,A Hard Day's Night,UK: A Hard Day's Night US: 1962-1966,"[Rock, Electronic, Pop/Rock]"
4,A Taste of Honey,UK: Please Please Me US: The Early Beatles,"[Pop/Rock, Jazz, Stage&Screen]"


## 2. Multiple Variables in One Column: UK and US Album Debuts

In the 1960s the Beatles' records were packaged differently in Britain and America. Many songs debuted on one album in the UK and a *different* album in the US. How many entries record two different releases?

In [18]:
beatles = beatles_clean.copy()
beatles['album_debut'].str.startswith('UK:').sum()

95

### Solution:  Split with `str.split(..., expand=True)`

The UK album always comes before `' US: '`, and the US album after it. `str.split(' US: ', expand=True)` splits each string at that point and puts the pieces in **separate columns**, which we assign to two new column names.

In [19]:
beatles[['album_debut_uk', 'album_debut_us']] = beatles['album_debut'].str.split(' US: ', expand=True)
beatles[['title', 'album_debut', 'album_debut_uk', 'album_debut_us']].iloc[[1, 2, 4]]

,title,album_debut,album_debut_uk,album_debut_us
1,A Day in the Life,Sgt. Pepper's Lonely Hearts Club Band,Sgt. Pepper's Lonely Hearts Club Band,None
2,A Hard Day's Night,UK: A Hard Day's Night US: 1962-1966,UK: A Hard Day's Night,1962-1966
4,A Taste of Honey,UK: Please Please Me US: The Early Beatles,UK: Please Please Me,The Early Beatles


Two problems remain:

1. The UK column still starts with `'UK: '`.
2. Where there was only one album (same release in both countries), the US column is empty (`None`).

### Removing a prefix: beware of `str.strip()`!

It is tempting to write `.str.strip('UK: ')`. But `strip()` does **not** remove the *word* `'UK: '`: it removes any of the *characters* `U`, `K`, `:`, and space from **both ends** of the string. Watch what happens to `'Unreleased'`:

In [20]:
'Unreleased'.strip('UK: ')

'nreleased'

Oops! The safe tool is `str.removeprefix()`, which removes an exact string only if it appears at the start:

In [21]:
beatles['album_debut_uk'] = beatles['album_debut_uk'].str.removeprefix('UK: ')
beatles[['title', 'album_debut_uk', 'album_debut_us']].iloc[[1, 2, 4]]

,title,album_debut_uk,album_debut_us
1,A Day in the Life,Sgt. Pepper's Lonely Hearts Club Band,None
2,A Hard Day's Night,A Hard Day's Night,1962-1966
4,A Taste of Honey,Please Please Me,The Early Beatles


### Filling the US column from the UK column

Where the US album is missing, the release was the same in both countries. `fillna()` can take another **column** as its argument; it fills each missing value with the value *from the same row* of that column.

In [22]:
beatles['album_debut_us'] = beatles['album_debut_us'].fillna(beatles['album_debut_uk'])
beatles[['title', 'album_debut_uk', 'album_debut_us']].iloc[[1, 2, 4]]


,title,album_debut_uk,album_debut_us
1,A Day in the Life,Sgt. Pepper's Lonely Hearts Club Band,Sgt. Pepper's Lonely Hearts Club Band
2,A Hard Day's Night,A Hard Day's Night,1962-1966
4,A Taste of Honey,Please Please Me,The Early Beatles


Finally, we drop the original combined column and put the two new ones where it used to be:

In [23]:
position = beatles.columns.get_loc('album_debut')
beatles = beatles.drop(columns=['album_debut'])

columns = list(beatles.columns)
columns.remove('album_debut_uk')
columns.remove('album_debut_us')
columns[position:position] = ['album_debut_uk', 'album_debut_us']
beatles = beatles[columns]

beatles.head(3)

,title,year,album,album_debut_uk,album_debut_us,songwriter,songwriter_group,lead_vocal,is_cover,genre,styles,themes,moods,popularity,duration,key,key_name,mode,mode_name,tempo,time_signature,valence,danceability,energy,loudness,acousticness,instrumentalness,liveness,speechiness,other_releases,single_a_side,single_b_side,single_certification,covered_by,billboard_top50,chart_position_uk_wikipedia,chart_position_us_wikipedia,highest_position_the_guardian,weeks_on_chart_in_uk_the_guardian,weeks_at_no1_in_uk_the_guardian,highest_position_billboard,weeks_at_no1_billboard,top_50_billboard,top_50_ultimate_classic_rock,top_50_rolling_stone,top_50_nme,top_50_top50songsorg,top_50_usa_today_2017,top_50_vulture_by_bill_wyman,uri
0,12-bar Original,1965,Anthology 2,Anthology 2,Anthology 2,"Lennon, McCartney, Harrison and Starkey",Whole Band,None (instrumental),False,[Blues],[],[],[Passionate],31,175,9,A,1,major,122.678,4,0.566,0.600,0.535,-8.178,0.00154,0.102000,0.1230,0.0298,<NA>,NaN,NaN,NaN,<NA>,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,spotify:track:2HvTGx5fzFGpHSyRNvXd9T
1,A Day in the Life,1967,Sgt. Pepper's Lonely Hearts Club Band,Sgt. Pepper's Lonely Hearts Club Band,Sgt. Pepper's Lonely Hearts Club Band,Lennon and McCartney,Lennon & McCartney,Lennon and McCartney,False,"[Psychedelic Rock, Art Rock, Pop/Rock]","[British Psychedelia, Contemporary Pop/Rock, E...",[],"[Ambitious, Complex, Dramatic, Dreamy, Eerie, ...",65,335,4,E,0,minor,163.219,4,0.175,0.364,0.457,-14.162,0.29000,0.000106,0.9220,0.0675,12,Sgt. Pepper's Lonely Hearts Club Band / With a...,A Day in the Life,NaN,27,False,63.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,2.0,18.0,1.0,1.0,spotify:track:0hKRSZhUGEhKU6aNSPBACZ
2,A Hard Day's Night,1964,A Hard Day's Night,A Hard Day's Night,1962-1966,Lennon,Lennon,"Lennon, with McCartney",False,"[Rock, Electronic, Pop/Rock]","[British Invasion, Contemporary Pop/Rock, Earl...","[Everyday Life, In Love, Relationships]","[Carefree, Cheerful, Explosive, Marching, Opti...",71,152,0,C,1,major,138.514,4,0.797,0.590,0.805,-6.481,0.13700,0.000000,0.0996,0.0371,35,A Hard Day's Night; A Hard Day's Night,Things We Said Today; I Should Have Known Better,RIAA Gold,35,True,1.0,1.0,1.0,13.0,3.0,1.0,2.0,8.0,18.0,11.0,19.0,19.0,15.0,41.0,spotify:track:5J2CHimS7dWYMImCHkEFaJ


### A first look: what did American listeners hear?

Now that UK and US releases are separate variables, questions become easy to ask. For instance, which American albums collected the songs from the British *Help!* and *Rubber Soul*?

In [24]:
uk_help_rubber = beatles[beatles['album_debut_uk'].isin(['Help!', 'Rubber Soul'])]
uk_help_rubber.groupby('album_debut_uk')['album_debut_us'].value_counts()

album_debut_uk  album_debut_us     
Help!           Help!                   7
                Beatles VI              3
                Rubber Soul             2
                Yesterday and Today     2
Rubber Soul     Rubber Soul            10
                Yesterday and Today     4
Name: count, dtype: int64

## 3. Multiple Observations in One Row: Genre (`explode`)

| Feature | `explode()` | `melt()` |
|---|---|---|
| **What it does** | Splits **list-like values** in one column into multiple rows | "Unpivots" several **columns** into rows |
| **Transformation** | list → rows | wide (columns) → long (rows) |
| **Our example** | `genre`: `['Rock', 'Pop/Rock']` | `energy`, `valence`, `danceability`, ... |
| **Useful for** | filtering, grouping, counting individual labels | grouping by variable, **plotting** |

The steps for `explode` are:

1. **Clean** the column: no missing values (Notebook A).
2. **Turn** each cell into a Python **list**. We already did this in Notebook A with `apply()`.
3. **Explode** the list into one row per item.

So our `genre` column is ready to go:

In [10]:
beatles[['title', 'genre']].head()

,title,genre
0,12-bar Original,[Blues]
1,A Day in the Life,"[Psychedelic Rock, Art Rock, Pop/Rock]"
2,A Hard Day's Night,"[Rock, Electronic, Pop/Rock]"
3,A Shot of Rhythm and Blues,"[R&B, Pop/Rock]"
4,A Taste of Honey,"[Pop/Rock, Jazz, Stage&Screen]"


### 3. Explode Genre Lists to Tidy Them

In our original dataset, the contents of the `genre` column were formatted as a single long string, such as `"Rock, Eletronic, Pop/Rock"`.  And so one of our Data Cleaning steps involved the use of `split` to transform these long strings into a `list` of tags:  `[Rock, Electronic, Pop/Rock]`.

Now as part of our Tidy Data protocol we will `explode` these lists:   `explode('genre')` makes one row per genre. Every other column is copied into each new row. Exploded rows keep their **old index number**, so we reset the index to give each row a unique number.

In [25]:
beatles_tidy = beatles.explode('genre').reset_index(drop=True)

print('songs before:', len(beatles))
print('rows after: ', len(beatles_tidy))

songs before: 278
rows after:  613


Here is "A Day in the Life" before the `explode()` step:

In [26]:
beatles[beatles['title'] == 'A Day in the Life'][['title', 'year', 'songwriter_group', 'genre']]

,title,year,songwriter_group,genre
1,A Day in the Life,1967,Lennon & McCartney,"[Psychedelic Rock, Art Rock, Pop/Rock]"


And here it is after the `explode` step:

In [27]:
beatles_tidy[beatles_tidy['title'] == 'A Day in the Life'][['title', 'year', 'songwriter_group', 'genre']]

,title,year,songwriter_group,genre
1,A Day in the Life,1967,Lennon & McCartney,Psychedelic Rock
2,A Day in the Life,1967,Lennon & McCartney,Art Rock
3,A Day in the Life,1967,Lennon & McCartney,Pop/Rock


Now we can count individual genres directly:

In [28]:
beatles_tidy['genre'].value_counts().head(15)

genre
Pop/Rock            273
Rock and Roll        45
Rock                 39
Psychedelic Rock     28
Folk Rock            21
Hard Rock            20
R&B                  16
Blues Rock           11
Baroque Pop          10
Merseybeat           10
Folk                  9
Country               8
Music Hall            8
Psychedelic Pop       6
Power Pop             5
Name: count, dtype: int64

> **Notice `Pop/Rock`.** It is attached to almost every song, so it tells us very little about any particular one. It is an *umbrella* label from the source data. In Notebooks C and D we will often leave it out to see the more interesting genres.

### Caution: counting songs in tidy data

After exploding, **one row ≠ one song**. A song with three genres now has three rows, so it is counted three times. Compare the number of songs by each songwriter in the clean data and in the tidy data:

In [29]:
print('rows in tidy data:', len(beatles_tidy), '   distinct songs:', beatles_tidy['title'].nunique())

pd.DataFrame({
    'rows in clean data': beatles_clean['songwriter_group'].value_counts(),
    'rows in tidy data': beatles_tidy['songwriter_group'].value_counts(),
    'distinct songs in tidy data': beatles_tidy.groupby('songwriter_group')['title'].nunique()
})

rows in tidy data: 613    distinct songs: 278


,rows in clean data,rows in tidy data,distinct songs in tidy data
songwriter_group,,,
Cover,69,124,69
Harrison,27,63,27
Lennon,77,176,77
Lennon & Harrison,1,2,1
Lennon & McCartney,17,40,17
McCartney,81,193,81
McCartney & Harrison,1,4,1
Starkey,2,4,2
Whole Band,3,7,3


Counting rows in the tidy data more than doubles every songwriter's output! Averages can be distorted the same way: songs with many genre labels get more "weight" than songs with just one. The fix is to count **distinct** titles with `.nunique()`, as in the last column.

The rule of thumb:

* Use the **tidy (exploded)** data when your question is about the **labels** ("How many songs are Psychedelic Rock?", "What is the average energy of Folk Rock songs?").
* Use **one row per song** when your question is about the **songs** as a whole ("What is the average energy of Lennon's songs?"). You can always get back to one row per song with `beatles_tidy.drop_duplicates(subset='title')`.

In [30]:
beatles_tidy.drop_duplicates(subset='title').shape

(278, 50)

## 4. Caution:  Why Not Explode Everything?

`styles`, `themes`, and `moods` are also multi-label columns. Why not explode them too?

Each explosion **multiplies** the rows. A song with 2 genres, 4 styles, and 12 moods becomes 2 × 4 × 12 = **96 rows**. Let's measure it:

In [17]:
step_1 = beatles_clean.explode('genre')
step_2 = step_1.explode('styles')
step_3 = step_2.explode('moods')

pd.DataFrame({
    'exploded columns': ['none (clean data)', 'genre', 'genre + styles', 'genre + styles + moods'],
    'rows': [len(beatles_clean), len(step_1), len(step_2), len(step_3)]
})

,exploded columns,rows
0,none (clean data),278
1,genre,613
2,genre + styles,2660
3,genre + styles + moods,37130


From 278 songs to tens of thousands of rows, almost all of them repeated information! This is slow, wasteful, and very easy to misuse (every average and count would be distorted).

**Best practice:** keep the clean, one-row-per-song data as your "master copy," and explode **one** label column at a time, only when a question calls for it. Because the labels are already lists, that's a one-liner. For example, the most common moods:

In [18]:
beatles_moods = beatles_clean.explode('moods')
beatles_moods['moods'].value_counts().head(10)

moods
Rousing         109
Rollicking      100
Freewheeling     97
Playful          95
Whimsical        82
Bright           78
Yearning         78
Energetic        74
Quirky           72
Cheerful         69
Name: count, dtype: int64

## 5. Wide to Long: Audio Features (`melt`)

Our Spotify audio features are spread across several columns ("wide" format):

In [31]:
features = ['danceability', 'energy', 'valence', 'acousticness', 'speechiness', 'liveness']

beatles_clean[beatles_clean['album'] == 'Rubber Soul'][['title'] + features].head(3)

,title,danceability,energy,valence,acousticness,speechiness,liveness
55,Drive My Car,0.754,0.571,0.820,0.01730,0.0421,0.3000
69,Girl,0.699,0.294,0.893,0.58700,0.0294,0.1170
117,I'm Looking Through You,0.667,0.498,0.774,0.00772,0.0364,0.0481


Many charting tools (especially Plotly Express, which we use in Notebook D) prefer **long** format: one row per song *per feature*. `pd.melt()` does this:

* `id_vars`: the column(s) that identify each row, which stay fixed
* `value_vars`: the columns to "unpivot" into rows
* `var_name` / `value_name`: names for the two new columns

In [32]:
rubber_soul = beatles_clean[beatles_clean['album'] == 'Rubber Soul']

rubber_soul_long = pd.melt(rubber_soul,
                           id_vars=['title'],
                           value_vars=features,
                           var_name='feature',
                           value_name='value')

rubber_soul_long.sort_values('title').head(12)

,title,feature,value
0,Drive My Car,danceability,0.7540
28,Drive My Car,valence,0.8200
56,Drive My Car,speechiness,0.0421
42,Drive My Car,acousticness,0.0173
14,Drive My Car,energy,0.5710
70,Drive My Car,liveness,0.3000
29,Girl,valence,0.8930
57,Girl,speechiness,0.0294
15,Girl,energy,0.2940
71,Girl,liveness,0.1170


Each song now takes 6 rows (one per feature). We'll use `melt` again in Notebook D to build **radar charts** and **line charts**. Because it is quick to do, we don't save melted data; we melt just the rows we need, when we need them.

## 6. Save the Tidy Data as a Pickle

Our tidy DataFrame has:

* separate `album_debut_uk` and `album_debut_us` columns, and
* one row per **song × genre**.

All the other columns (including the `styles`, `themes`, and `moods` lists) are unchanged, so we can still explode them later if needed.

In [33]:
beatles_tidy.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 613 entries, 0 to 612
Data columns (total 50 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   title                              613 non-null    object 
 1   year                               609 non-null    Int64  
 2   album                              613 non-null    object 
 3   album_debut_uk                     613 non-null    object 
 4   album_debut_us                     613 non-null    object 
 5   songwriter                         613 non-null    object 
 6   songwriter_group                   613 non-null    object 
 7   lead_vocal                         613 non-null    object 
 8   is_cover                           613 non-null    bool   
 9   genre                              613 non-null    object 
 10  styles                             613 non-null    object 
 11  themes                             613 non-null    object 

In [34]:
beatles_tidy.to_pickle('beatles_tidy_genre.pkl')

pd.read_pickle('beatles_tidy_genre.pkl').shape

(613, 50)

### Summary

| Tidy principle | Problem | Tool | Result |
|---|---|---|---|
| Each variable a column | UK & US album in one cell | `str.split(expand=True)`, `str.removeprefix()`, `fillna(other_column)` | `album_debut_uk`, `album_debut_us` |
| Each observation a row | Several genres per cell | lists (Notebook A) + `explode()` | one row per song × genre |
| (for plotting) | Features spread across columns | `pd.melt()` | one row per song × feature |

**Next:** in **Notebook C** we use both `beatles_clean.pkl` and `beatles_tidy_genre.pkl` to **find, filter, and group** our data.

**Beatles notebooks:** [A. Clean Data](A_Beatles_Clean_Data.ipynb) · **B. Tidy Data** · [C. Find, Filter, Group](C_Beatles_Find_Filter_Group.ipynb) · [D. Charts and Graphs](D_Beatles_Charts_Graphs.ipynb)